<a href="https://colab.research.google.com/github/gee666thu/r5-machine-learning-lab/blob/main/modf%20exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

In [11]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']
print("Loading 20 Newsgroups dataset...")
train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers','footers','quotes'))
test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers','footers','quotes'))
vectorizer = CountVectorizer(stop_words='english', max_features=5000)
X_train = vectorizer.fit_transform(train.data).toarray()

Loading 20 Newsgroups dataset...


In [12]:
y_train = train.target
X_test = vectorizer.transform(test.data).toarray()
y_test = test.target
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (2257, 5000), Test shape: (1502, 5000)


In [13]:
n_classes = len(categories)
n_features = X_train.shape[1]
class_counts = np.bincount(y_train)
p_labels = class_counts / len(y_train)
theta = np.zeros((n_classes, n_features))

In [14]:
for c in range(n_classes):
    theta[c, :] = X_train[y_train == c].sum(axis=0)
print("Word counts aggregated successfully.")

Word counts aggregated successfully.


In [15]:
with np.errstate(divide='ignore'):
    log_theta = np.log(theta)
log_theta[np.isinf(log_theta)] = -100

In [16]:
posterior = X_test @ log_theta.T + np.log(p_labels)
y_pred_mle = np.argmax(posterior, axis=1)
acc_mle = accuracy_score(y_test, y_pred_mle)
print(f"[MLE] Test Accuracy: {acc_mle:.4f}")

[MLE] Test Accuracy: 0.7423


In [17]:
print("Evaluating MAP Predictions:")

Evaluating MAP Predictions:


In [18]:
priors = {
    "Lidstone Smoothing (Alpha = 1.01)": np.ones(n_features) * 1.01,
    "Laplace Smoothing (Alpha = 2.0)": np.ones(n_features) * 2.0,
    "Strong Dirichlet Prior (Alpha = 10.0)": np.ones(n_features) * 10.0,
    "Non-Uniform Prior (Empirical Basis)": 1.0 + (X_train.sum(axis=0) / X_train.sum() * 100)
}

In [20]:
for name, alpha in priors.items():
    numerator = theta + alpha - 1
    denominator = numerator.sum(axis=1, keepdims=True)
    theta_map = numerator / denominator
    y_pred = np.argmax(X_test @ np.log(theta_map).T + np.log(p_labels), axis=1)
    acc = accuracy_score(y_test, y_pred)
    print(name, "Test Accuracy:", round(acc, 4))

Lidstone Smoothing (Alpha = 1.01) Test Accuracy: 0.8103
Laplace Smoothing (Alpha = 2.0) Test Accuracy: 0.8182
Strong Dirichlet Prior (Alpha = 10.0) Test Accuracy: 0.7863
Non-Uniform Prior (Empirical Basis) Test Accuracy: 0.8063
